
# 05. PepBERT Training

## Objective

Fine-tune a pretrained protein language model for peptide MIC prediction and compare its performance with traditional machine learning models.

## Experimental Design

- Datasets: Escherichia coli (EC), Staphylococcus aureus (SA)
- Input: Peptide amino acid sequences
- Target: pMIC
- Task: Regression
- Random seeds: 42, 43, 44, 45, 46
- Hyperparameter selection: Validation MSE
- Evaluation: Independent test dataset

## Workflow

1. Initialize the training environment.
2. Load pretrained model and tokenizer.
3. Prepare sequence datasets and data loaders.
4. Implement the regression model.
5. Optimize hyperparameters using five validation splits.
6. Train final models and evaluate test datasets.
7. Save model checkpoints, hyperparameters, predictions, and evaluation metrics.


In [1]:

import gc
import importlib
import torch
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset

from configs.common import (
    ROOT,
    DATA_DIR,
    MODEL_DIR,
    RESULT_DIR,
    RANDOM_STATE,
    RANDOM_SEEDS
)

from configs import pepbert as pepbert_config
from src import pepbert as pepbert_module
from src import pepbert_pretrained as pretrained_module

from src.seed import set_seed
from src.ml import load_validation_split

# Reload project modules
importlib.reload(pepbert_config)
importlib.reload(pepbert_module)
importlib.reload(pretrained_module)

# Define datasets
TARGETS = ["EC", "SA"]

# Set random seed
set_seed(RANDOM_STATE)

# Set device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Check environment
print("Project:", ROOT)
print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Datasets:", TARGETS)
print("Seeds:", RANDOM_SEEDS)
print("Pooling:", pepbert_config.POOLING_TYPES)
print("FC Layers:", pepbert_config.FC_LAYERS)

print("\n[OK] Environment initialized.")


Project: C:\Users\jinwo\Desktop\PepBert_REV
PyTorch: 2.13.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti
Datasets: ['EC', 'SA']
Seeds: [42, 43, 44, 45, 46]
Pooling: ['sos', 'mean']
FC Layers: [1, 2, 3, 4]

[OK] Environment initialized.


In [2]:

# Load pretrained PepBERT
model_module, pretrained_config, tokenizer, pretrained_weights = (
    pretrained_module.load_pretrained_pepbert()
)

# Initialize pretrained encoder
encoder = pretrained_module.build_pretrained_encoder(
    model_module=model_module,
    pretrained_config=pretrained_config,
    tokenizer=tokenizer,
    pretrained_weights=pretrained_weights
)

# Check pretrained model
print("Model: PepBERT-large-UniParc")
print("Vocabulary Size:", tokenizer.get_vocab_size())
print("Sequence Length:", pretrained_config["seq_len"])
print("Hidden Size:", pretrained_config["d_model"])
print("Parameters:", f"{sum(p.numel() for p in encoder.parameters()):,}")

# Check special tokens
print("SOS ID:", tokenizer.token_to_id("[SOS]"))
print("EOS ID:", tokenizer.token_to_id("[EOS]"))
print("PAD ID:", tokenizer.token_to_id("[PAD]"))

print("\n[OK] Pretrained PepBERT loaded.")


Model: PepBERT-large-UniParc
Vocabulary Size: 29
Sequence Length: 52
Hidden Size: 320
Parameters: 4,947,869
SOS ID: 2
EOS ID: 3
PAD ID: 1

[OK] Pretrained PepBERT loaded.


In [3]:
# Load EC and SA training data
train_data = {
    target: pd.read_csv(DATA_DIR / f"{target}_Train.csv")
    for target in TARGETS
}

# Create tokenized datasets
train_datasets = {
    target: pepbert_module.PeptideDataset(
        train_data[target],
        tokenizer,
        max_length=pepbert_config.MAX_LENGTH
    )
    for target in TARGETS
}

# Check datasets
for target in TARGETS:
    df = train_data[target]
    dataset = train_datasets[target]
    sample = dataset[0]

    print(f"\n[{target}]")
    print("Samples:", len(dataset))
    print("Sequence:", df.iloc[0]["sequence"])
    print("Input IDs:", sample["input_ids"].shape)
    print("Attention Mask:", sample["attention_mask"].shape)
    print("Label (pMIC):", sample["labels"].item())

print("\n[OK] EC / SA datasets loaded.")


[EC]
Samples: 7583
Sequence: AAAAAAAIKMLMDLVNERIMALNKKAKK
Input IDs: torch.Size([52])
Attention Mask: torch.Size([52])
Label (pMIC): 5.0

[SA]
Samples: 6592
Sequence: AAAAAAAIKMLMDLVNERIMALNKKAKK
Input IDs: torch.Size([52])
Attention Mask: torch.Size([52])
Label (pMIC): 4.698969841003418

[OK] EC / SA datasets loaded.


In [4]:
# Prepare sample input
sample_loader = DataLoader(
    train_datasets["EC"],
    batch_size=2,
    shuffle=False
)

batch = next(iter(sample_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

records = []

# Validate all architectures
for pooling in pepbert_config.POOLING_TYPES:
    for fc_layers in pepbert_config.FC_LAYERS:

        # Initialize fresh pretrained encoder
        encoder = pretrained_module.build_pretrained_encoder(
            model_module=model_module,
            pretrained_config=pretrained_config,
            tokenizer=tokenizer,
            pretrained_weights=pretrained_weights
        )

        # Create regression model
        model = pepbert_module.PepBERTRegressor(
            encoder=encoder,
            hidden_size=pretrained_config["d_model"],
            pooling=pooling,
            fc_layers=fc_layers,
            dropout=pepbert_config.DROPOUT
        ).to(device)

        # Test forward pass
        model.eval()

        with torch.no_grad():
            predictions = model(input_ids, attention_mask)

        assert predictions.shape == (2,)

        records.append({
            "Pooling": pooling,
            "FC_Layers": fc_layers,
            "Output_Shape": str(tuple(predictions.shape)),
            "Status": "OK"
        })

        del model, encoder

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Display architecture validation
architecture_check = pd.DataFrame(records)

display(architecture_check)

print("\n[OK] All 8 architectures validated.")

,Pooling,FC_Layers,Output_Shape,Status
0,sos,1,"(2,)",OK
1,sos,2,"(2,)",OK
2,sos,3,"(2,)",OK
3,sos,4,"(2,)",OK
4,mean,1,"(2,)",OK
5,mean,2,"(2,)",OK
6,mean,3,"(2,)",OK
7,mean,4,"(2,)",OK



[OK] All 8 architectures validated.


In [5]:
# Check train/validation splits
split_records = []

for target in TARGETS:
    dataset = train_datasets[target]

    for seed in RANDOM_SEEDS:

        # Load existing split
        train_idx, valid_idx = load_validation_split(
            target, seed
        )

        # Validate indices
        assert len(train_idx) + len(valid_idx) == len(dataset)
        assert set(train_idx).isdisjoint(set(valid_idx))

        split_records.append({
            "Dataset": target,
            "Seed": seed,
            "Train": len(train_idx),
            "Validation": len(valid_idx),
            "Total": len(dataset),
            "Status": "OK"
        })

# Display split summary
split_summary = pd.DataFrame(split_records)

display(split_summary)

print("\n[OK] All EC / SA validation splits verified.")

,Dataset,Seed,Train,Validation,Total,Status
0,EC,42,6824,759,7583,OK
1,EC,43,6824,759,7583,OK
2,EC,44,6824,759,7583,OK
3,EC,45,6824,759,7583,OK
4,EC,46,6824,759,7583,OK
5,SA,42,5932,660,6592,OK
6,SA,43,5932,660,6592,OK
7,SA,44,5932,660,6592,OK
8,SA,45,5932,660,6592,OK
9,SA,46,5932,660,6592,OK



[OK] All EC / SA validation splits verified.


In [6]:
import importlib
import inspect

import src.pepbert as pepbert_module
import src.pepbert_search as search_module

# Update tqdm display settings
module_path = ROOT / "src" / "pepbert.py"
source = module_path.read_text(encoding="utf-8")

old = """        position=1,
        leave=False,
        ncols=150"""

new = """        position=0,
        leave=True,
        ncols=150,
        mininterval=1"""

if old in source:
    source = source.replace(old, new, 1)
    module_path.write_text(source, encoding="utf-8")
    print("[OK] tqdm display settings updated.")
elif new in source:
    print("[OK] tqdm settings already updated.")
else:
    print("[CHECK] tqdm settings not found. Review src/pepbert.py.")

# Reload training modules
importlib.reload(pepbert_module)
importlib.reload(search_module)

# Verify training function
print("Training function:", inspect.signature(
    pepbert_module.train_pepbert
))

print("Architecture Search:",
      hasattr(search_module, "run_architecture_search"))

[OK] tqdm settings already updated.
Training function: (model, train_loader, valid_loader, seed, save_path, history_path, max_epochs=100, patience=7, lr=1e-05, weight_decay=0.01, run_name='PepBERT')
Architecture Search: True


In [7]:
import importlib
import src.pepbert_search as search_module

# Reload architecture search module
importlib.reload(search_module)

# Run EC and SA architecture search
architecture_results = search_module.run_architecture_search(
    targets=TARGETS,
    train_datasets=train_datasets,
    model_module=model_module,
    pretrained_config=pretrained_config,
    tokenizer=tokenizer,
    pretrained_weights=pretrained_weights,
    device=device
)

print("\n[OK] EC / SA Architecture Search completed.")

# Display results
display(architecture_results)

\n[Run 1/80] EC | SOS | FC1 | Seed 42


1/80 | EC | SOS | FC1 | Seed 42:  27%|██▋       | 27/100 [02:00<05:26,  4.48s/epoch, Train=0.0457, Val=0.3104, Best=0.2966, BestEp=18, Patience=10/10]

[Completed 1/80] Best Epoch=18 | Best Val MSE=0.296574
\n[Run 2/80] EC | SOS | FC1 | Seed 43



2/80 | EC | SOS | FC1 | Seed 43:  26%|██▌       | 26/100 [01:52<05:19,  4.32s/epoch, Train=0.0576, Val=0.3255, Best=0.3108, BestEp=17, Patience=10/10]

[Completed 2/80] Best Epoch=17 | Best Val MSE=0.310792
\n[Run 3/80] EC | SOS | FC1 | Seed 44



3/80 | EC | SOS | FC1 | Seed 44:  29%|██▉       | 29/100 [02:01<04:57,  4.19s/epoch, Train=0.0367, Val=0.3116, Best=0.2791, BestEp=20, Patience=10/10]

... (152 outputs left out)

[Completed 79/80] Best Epoch=31 | Best Val MSE=0.291566
\n[Run 80/80] SA | MEAN | FC4 | Seed 46



80/80 | SA | MEAN | FC4 | Seed 46:  40%|███▏    | 40/100 [02:32<03:48,  3.81s/epoch, Train=0.5295, Val=0.3029, Best=0.2882, BestEp=31, Patience=10/10]

[Completed 80/80] Best Epoch=31 | Best Val MSE=0.288175

[OK] EC / SA Architecture Search completed.


,Dataset,Pooling,FC_Layers,Seed,Dropout,Learning_Rate,Weight_Decay,Best_Epoch,Validation_MSE
0,EC,sos,1,42,0.2,0.00001,0.01,18,0.296574
1,EC,sos,1,43,0.2,0.00001,0.01,17,0.310792
2,EC,sos,1,44,0.2,0.00001,0.01,20,0.279074
3,EC,sos,1,45,0.2,0.00001,0.01,17,0.250020
4,EC,sos,1,46,0.2,0.00001,0.01,16,0.260188
...,...,...,...,...,...,...,...,...,...
75,SA,mean,4,42,0.2,0.00001,0.01,69,0.298199
76,SA,mean,4,43,0.2,0.00001,0.01,37,0.292167
77,SA,mean,4,44,0.2,0.00001,0.01,37,0.304783
78,SA,mean,4,45,0.2,0.00001,0.01,31,0.291566


In [8]:
import pandas as pd

from configs.common import RESULT_DIR

# Define result directory
result_dir = RESULT_DIR / "pepbert"

# Load EC and SA results
results = pd.concat(
    [
        pd.read_csv(
            result_dir / target / "architecture_search_results.csv"
        )
        for target in TARGETS
    ],
    ignore_index=True
)

# Summarize validation performance
summary = (
    results
    .groupby(["Dataset", "Pooling", "FC_Layers"])
    .agg(
        Mean_Validation_MSE=("Validation_MSE", "mean"),
        SD_Validation_MSE=("Validation_MSE", "std"),
        Mean_Best_Epoch=("Best_Epoch", "mean"),
        N_Seeds=("Seed", "nunique")
    )
    .reset_index()
    .sort_values(
        ["Dataset", "Pooling", "Mean_Validation_MSE"]
    )
)

# Verify completed experiments
assert len(results) == 80, "Architecture Search is incomplete."
assert (summary["N_Seeds"] == 5).all(), "Missing random seeds."

# Save architecture comparison
summary.to_csv(
    result_dir / "architecture_search_summary.csv",
    index=False
)

# Select best FC architecture for each dataset and pooling
best_architectures = (
    summary
    .sort_values("Mean_Validation_MSE")
    .groupby(["Dataset", "Pooling"], as_index=False)
    .first()
    .sort_values(["Dataset", "Pooling"])
    .reset_index(drop=True)
)

# Save selected architectures
best_architectures.to_csv(
    result_dir / "best_architectures.csv",
    index=False
)

print("[OK] Architecture Search completed.")
print("[OK] Best FC architectures selected.")

print("\n[Architecture Comparison]")
display(summary.round(6))

print("\n[Best FC Architectures]")
display(best_architectures.round(6))

[OK] Architecture Search completed.
[OK] Best FC architectures selected.

[Architecture Comparison]


,Dataset,Pooling,FC_Layers,Mean_Validation_MSE,SD_Validation_MSE,Mean_Best_Epoch,N_Seeds
2,EC,mean,3,0.288981,0.032495,32.8,5
1,EC,mean,2,0.293615,0.034831,25.0,5
3,EC,mean,4,0.293924,0.030533,41.2,5
0,EC,mean,1,0.294577,0.027594,22.6,5
4,EC,sos,1,0.279330,0.025064,17.6,5
6,EC,sos,3,0.285144,0.031388,19.8,5
5,EC,sos,2,0.288562,0.032886,18.2,5
7,EC,sos,4,0.288745,0.029916,26.0,5
8,SA,mean,1,0.290290,0.013392,23.2,5
9,SA,mean,2,0.291393,0.007766,28.4,5



[Best FC Architectures]


,Dataset,Pooling,FC_Layers,Mean_Validation_MSE,SD_Validation_MSE,Mean_Best_Epoch,N_Seeds
0,EC,mean,3,0.288981,0.032495,32.8,5
1,EC,sos,1,0.279330,0.025064,17.6,5
2,SA,mean,1,0.290290,0.013392,23.2,5
3,SA,sos,2,0.281040,0.004472,20.0,5


In [9]:
import importlib
import torch
import pandas as pd

from configs.common import (
    ROOT,
    DATA_DIR,
    MODEL_DIR,
    RESULT_DIR,
    RANDOM_STATE,
    RANDOM_SEEDS
)

from configs import pepbert as pepbert_config
from src import pepbert as pepbert_module
from src import pepbert_pretrained as pretrained_module
from src.seed import set_seed

# Reload modules
importlib.reload(pepbert_config)
importlib.reload(pepbert_module)
importlib.reload(pretrained_module)

# Set environment
TARGETS = ["EC", "SA"]
set_seed(RANDOM_STATE)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load selected architectures from Notebook 05
best_architectures = pd.read_csv(
    RESULT_DIR / "pepbert" / "best_architectures.csv"
)

# Verify selected architectures
assert len(best_architectures) == 4
assert (best_architectures["N_Seeds"] == len(RANDOM_SEEDS)).all()

print("Device:", device)
print("Selected architectures:")
display(
    best_architectures[
        [
            "Dataset",
            "Pooling",
            "FC_Layers",
            "Mean_Validation_MSE",
            "SD_Validation_MSE"
        ]
    ].round(6)
)

print("\n[OK] Best architectures loaded.")

Device: cuda
Selected architectures:


,Dataset,Pooling,FC_Layers,Mean_Validation_MSE,SD_Validation_MSE
0,EC,mean,3,0.288981,0.032495
1,EC,sos,1,0.279330,0.025064
2,SA,mean,1,0.290290,0.013392
3,SA,sos,2,0.281040,0.004472



[OK] Best architectures loaded.


In [10]:
import importlib
import pandas as pd
from itertools import product

import configs.pepbert_grid as grid_config

# Reload grid search configuration
importlib.reload(grid_config)

# Create grid search tasks
grid_tasks = []

for row in best_architectures.itertuples(index=False):

    dropouts = (
        [0.0] if row.FC_Layers == 1
        else grid_config.DROPOUTS
    )

    for lr, wd, dropout in product(
        grid_config.LEARNING_RATES,
        grid_config.WEIGHT_DECAYS,
        dropouts
    ):
        grid_tasks.append({
            "Dataset": row.Dataset,
            "Pooling": row.Pooling,
            "FC_Layers": int(row.FC_Layers),
            "Learning_Rate": lr,
            "Weight_Decay": wd,
            "Dropout": dropout
        })

# Summarize grid search
grid_summary = (
    pd.DataFrame(grid_tasks)
    .groupby(["Dataset", "Pooling", "FC_Layers"])
    .size()
    .reset_index(name="Combinations")
)

grid_summary["Runs"] = (
    grid_summary["Combinations"] * len(RANDOM_SEEDS)
)

print("[Grid Search Summary]")
display(grid_summary)

print("Total Hyperparameter Combinations:", len(grid_tasks))
print("Total Fine-tuning Runs:", grid_summary["Runs"].sum())
print("Random Seeds:", RANDOM_SEEDS)
print("Early Stopping Patience:", grid_config.PATIENCE)

print("\n[OK] Grid Search tasks prepared.")

[Grid Search Summary]


,Dataset,Pooling,FC_Layers,Combinations,Runs
0,EC,mean,3,24,120
1,EC,sos,1,6,30
2,SA,mean,1,6,30
3,SA,sos,2,24,120


Total Hyperparameter Combinations: 60
Total Fine-tuning Runs: 300
Random Seeds: [42, 43, 44, 45, 46]
Early Stopping Patience: 10

[OK] Grid Search tasks prepared.


In [11]:
# Load pretrained model
model_module, pretrained_config, tokenizer, pretrained_weights = (
    pretrained_module.load_pretrained_pepbert()
)

# Load EC and SA training datasets
train_datasets = {
    target: pepbert_module.PeptideDataset(
        pd.read_csv(DATA_DIR / f"{target}_Train.csv"),
        tokenizer,
        max_length=pepbert_config.MAX_LENGTH
    )
    for target in TARGETS
}

print("Pretrained PepBERT and datasets loaded.")
print("EC:", len(train_datasets["EC"]))
print("SA:", len(train_datasets["SA"]))

Pretrained PepBERT and datasets loaded.
EC: 7583
SA: 6592


In [12]:
import importlib
import src.pepbert_grid as grid_module

# Reload grid search module
importlib.reload(grid_module)

# Run hyperparameter optimization
grid_results = grid_module.run_grid_search(
    grid_tasks=grid_tasks,
    train_datasets=train_datasets,
    model_module=model_module,
    pretrained_config=pretrained_config,
    tokenizer=tokenizer,
    pretrained_weights=pretrained_weights,
    device=device
)

print("\n[OK] PepBERT Grid Search completed.")

display(grid_results)

1/300 | EC | MEAN | FC3 | LR=1e-05 | WD=1e-03 | DO=0.1 | Seed=42:  31%|▎| 31/100 [02:16<05:04,  4.42s/epoch, Train=0.1961, Val=0.3182, Best=0.3039, Be

[Completed 1/300] EC | MEAN | FC3 | Best Val=0.303924 | Best Epoch=22



2/300 | EC | MEAN | FC3 | LR=1e-05 | WD=1e-03 | DO=0.1 | Seed=43:  39%|▍| 39/100 [02:48<04:22,  4.31s/epoch, Train=0.1794, Val=0.3483, Best=0.3249, Be

[Completed 2/300] EC | MEAN | FC3 | Best Val=0.324866 | Best Epoch=30



3/300 | EC | MEAN | FC3 | LR=1e-05 | WD=1e-03 | DO=0.1 | Seed=44:  32%|▎| 32/100 [02:19<04:56,  4.36s/epoch, Train=0.2194, Val=0.3218, Best=0.3124, Be

[Completed 3/300] EC | MEAN | FC3 | Best Val=0.312366 | Best Epoch=23


... (591 outputs left out)

[Completed 299/300] SA | SOS | FC2 | Best Val=0.266911 | Best Epoch=7



300/300 | SA | SOS | FC2 | LR=1e-04 | WD=1e-02 | DO=0.4 | Seed=46:  26%|▎| 26/100 [01:41<04:48,  3.90s/epoch, Train=0.1861, Val=0.2520, Best=0.2489, B

[Completed 300/300] SA | SOS | FC2 | Best Val=0.248911 | Best Epoch=17

[OK] PepBERT Grid Search completed.


,Dataset,Pooling,FC_Layers,Learning_Rate,Weight_Decay,Dropout,Seed,Best_Epoch,Validation_MSE
0,EC,mean,3,0.00001,0.001,0.1,42,22,0.303924
1,EC,mean,3,0.00001,0.001,0.1,43,30,0.324866
2,EC,mean,3,0.00001,0.001,0.1,44,23,0.312366
3,EC,mean,3,0.00001,0.001,0.1,45,22,0.260497
4,EC,mean,3,0.00001,0.001,0.1,46,32,0.253561
...,...,...,...,...,...,...,...,...,...
295,SA,sos,2,0.00010,0.010,0.4,42,11,0.260102
296,SA,sos,2,0.00010,0.010,0.4,43,12,0.266297
297,SA,sos,2,0.00010,0.010,0.4,44,5,0.264207
298,SA,sos,2,0.00010,0.010,0.4,45,7,0.266911


In [13]:
import pandas as pd
from configs.common import RESULT_DIR, RANDOM_SEEDS

# Define result directory
result_dir = RESULT_DIR / "pepbert"

# Load grid search results
results = pd.read_csv(
    result_dir / "grid_search_results.csv"
)

# Verify completed experiments
expected_runs = len(grid_tasks) * len(RANDOM_SEEDS)

assert len(results) == expected_runs, (
    f"Incomplete Grid Search: {len(results)}/{expected_runs}"
)

# Summarize validation performance
summary = (
    results
    .groupby([
        "Dataset",
        "Pooling",
        "FC_Layers",
        "Learning_Rate",
        "Weight_Decay",
        "Dropout"
    ])
    .agg(
        Mean_Validation_MSE=("Validation_MSE", "mean"),
        SD_Validation_MSE=("Validation_MSE", "std"),
        Mean_Best_Epoch=("Best_Epoch", "mean"),
        N_Seeds=("Seed", "nunique")
    )
    .reset_index()
    .sort_values([
        "Dataset",
        "Pooling",
        "Mean_Validation_MSE"
    ])
)

# Verify five seeds per combination
assert (summary["N_Seeds"] == len(RANDOM_SEEDS)).all()

# Save all validation results
summary.to_csv(
    result_dir / "grid_search_summary.csv",
    index=False
)

# Select best hyperparameters for each dataset and pooling
best_params = (
    summary
    .sort_values("Mean_Validation_MSE")
    .groupby(["Dataset", "Pooling"], as_index=False)
    .first()
    .sort_values(["Dataset", "Pooling"])
    .reset_index(drop=True)
)

# Save selected hyperparameters
best_params.to_csv(
    result_dir / "best_hyperparameters.csv",
    index=False
)

print("[OK] Grid Search validation summary saved.")
print("[OK] Best hyperparameters selected.")

print("\n[Best Hyperparameters]")
display(best_params.round(6))

[OK] Grid Search validation summary saved.
[OK] Best hyperparameters selected.

[Best Hyperparameters]


,Dataset,Pooling,FC_Layers,Learning_Rate,Weight_Decay,Dropout,Mean_Validation_MSE,SD_Validation_MSE,Mean_Best_Epoch,N_Seeds
0,EC,mean,3,0.0001,0.01,0.1,0.262012,0.023592,21.4,5
1,EC,sos,1,0.0001,0.01,0.0,0.251818,0.022752,36.2,5
2,SA,mean,1,0.0001,0.01,0.0,0.271553,0.018978,21.8,5
3,SA,sos,2,0.0001,0.01,0.3,0.255774,0.010659,17.4,5
